# -----------------------------------------------------
# Libraries
# -----------------------------------------------------

In [17]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# -----------------------------------------------------
# PATH CONFIGURATION
# -----------------------------------------------------

In [18]:
# -----------------------------------------------------
# PATH CONFIGURATION
# -----------------------------------------------------

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

from config.paths import RAW_DIR, PROCESSED_DIR

YEARS = [2018, 2024]

# -----------------------------------------------------
# TAX BRACKETS BY YEAR
# -----------------------------------------------------

In [19]:
# -----------------------------------------------------
# TAX BRACKETS BY YEAR
# -----------------------------------------------------

TAX_BRACKETS = {
    2018: [
        {"li": 0.01, "ls": 8952.49, "cf": 0, "tm": 0.0192},
        {"li": 8952.50, "ls": 75984.55, "cf": 171.88, "tm": 0.064},
        {"li": 75984.56, "ls": 133536.07, "cf": 4461.94, "tm": 0.1088},
        {"li": 133536.08, "ls": 155229.80, "cf": 10723.55, "tm": 0.16},
        {"li": 155229.81, "ls": 185852.57, "cf": 14194.54, "tm": 0.1792},
        {"li": 185852.58, "ls": 374837.88, "cf": 19682.13, "tm": 0.2136},
        {"li": 374837.89, "ls": 590795.99, "cf": 60049.40, "tm": 0.2352},
        {"li": 590796.00, "ls": 1127926.84, "cf": 110842.74, "tm": 0.30},
        {"li": 1127926.85, "ls": 1503902.46, "cf": 271981.99, "tm": 0.32},
        {"li": 1503902.47, "ls": 4511707.37, "cf": 392294.17, "tm": 0.34},
        {"li": 4511707.38, "ls": np.inf, "cf": 1414947.85, "tm": 0.35},
    ],

    2024: [
        {"li": 0.01, "ls": 8952.49, "cf": 0, "tm": 0.0192},
        {"li": 8952.50, "ls": 75984.55, "cf": 171.88, "tm": 0.064},
        {"li": 75984.56, "ls": 133536.07, "cf": 4461.94, "tm": 0.1088},
        {"li": 133536.08, "ls": 155229.80, "cf": 10723.55, "tm": 0.16},
        {"li": 155229.81, "ls": 185852.57, "cf": 14194.54, "tm": 0.1792},
        {"li": 185852.58, "ls": 374837.88, "cf": 19682.13, "tm": 0.2136},
        {"li": 374837.89, "ls": 590795.99, "cf": 60049.40, "tm": 0.2352},
        {"li": 590796.00, "ls": 1127926.84, "cf": 110842.74, "tm": 0.30},
        {"li": 1127926.85, "ls": 1503902.46, "cf": 271981.99, "tm": 0.32},
        {"li": 1503902.47, "ls": 4511707.37, "cf": 392294.17, "tm": 0.34},
        {"li": 4511707.38, "ls": np.inf, "cf": 1414947.85, "tm": 0.35},
    ]
}

# -----------------------------------------------------
# FUNCTION: GROSS-UP CALCULATION
# -----------------------------------------------------

In [20]:
# -----------------------------------------------------
# FUNCTION: GROSS-UP CALCULATION
# -----------------------------------------------------

def calculate_market_income(df, brackets):

    df["market_income"] = np.nan

    for b in brackets:

        gross_temp = (
            df["net_market_income"]
            + b["cf"]
            - b["tm"] * b["li"]
        ) / (1 - b["tm"])

        mask = (
            df["market_income"].isna()
        ) & (
            gross_temp >= b["li"]
        ) & (
            gross_temp <= b["ls"]
        )

        df.loc[mask, "market_income"] = gross_temp

    return df

# -----------------------------------------------------
# MAIN PIPELINE
# -----------------------------------------------------

In [21]:
# =====================================================
# BLOQUE 1
# CONSTRUIR INGRESOS
# =====================================================

import pandas as pd

CURRENT_INCOME_KEYS = [
    # Trabajo
    "P001","P002","P003","P004","P005","P006","P007","P008","P009",
    "P011","P012","P013","P014","P015","P016","P018","P019","P020",
    "P021","P022",

    # Renta de propiedad
    "P023","P024","P025","P026","P027","P028","P029","P030","P031",

    # Transferencias
    "P032","P033","P037","P038","P039","P040","P041",
    "P042","P043","P044","P045","P046","P047","P048",

    # Negocios
    "P068","P069","P070","P071","P072","P073","P074",
    "P075","P076","P077","P078","P079","P080","P081"
]

TRANSFER_GOV_KEYS = [
    "P038","P042","P043","P044","P045","P046","P047","P048"
]

TRANSFER_ALL_KEYS = [
    "P032","P033","P037","P038","P039","P040","P041",
    "P042","P043","P044","P045","P046","P047","P048"
]

DEFLATORS = {
    2018: 1.0,
    2024: (100.0 / 133.543876114864)
}

for year in YEARS:

    print(f"\nProcessing income {year}")

    df_ing = pd.read_csv(
        RAW_DIR / f"ingresos_{year}.csv",
        dtype={"folioviv":"string","foliohog":"string","numren":"int64"},
        low_memory=False
    )

    # -----------------------------
    # DEFLACTAR
    # -----------------------------

    deflator = DEFLATORS.get(year, 1.0)
    df_ing["ing_tri_def"] = df_ing["ing_tri"] * deflator

    # -----------------------------
    # FILTRAR INGRESO CORRIENTE
    # -----------------------------

    df_ing = df_ing[df_ing["clave"].isin(CURRENT_INCOME_KEYS)].copy()

    # -----------------------------
    # CLASIFICAR COMPONENTES
    # -----------------------------

    df_ing["is_transfer_all"] = df_ing["clave"].isin(TRANSFER_ALL_KEYS)
    df_ing["is_transfer_gov"] = df_ing["clave"].isin(TRANSFER_GOV_KEYS)

    df_ing["transfer_all"] = df_ing.apply(
        lambda x: x["ing_tri_def"] if x["is_transfer_all"] else 0, axis=1
    )

    df_ing["transfer_gov"] = df_ing.apply(
        lambda x: x["ing_tri_def"] if x["is_transfer_gov"] else 0, axis=1
    )

    df_ing["market_component"] = df_ing.apply(
        lambda x: x["ing_tri_def"] if not x["is_transfer_all"] else 0, axis=1
    )

    # -----------------------------
    # AGREGAR A PERSONA
    # -----------------------------

    df_person = df_ing.groupby(
        ["folioviv","foliohog","numren"]
    ).agg(
        market_income=("market_component","sum"),
        current_income=("ing_tri_def","sum"),
        transfers_total=("transfer_all","sum"),
        transfers_gov=("transfer_gov","sum")
    ).reset_index()

    # -----------------------------
    # AGREGAR TRANSFERENCIAS A HOGAR
    # -----------------------------

    df_hh = df_person.groupby(
        ["folioviv","foliohog"]
    )["transfers_gov"].sum().reset_index()

    df_hh = df_hh.rename(columns={"transfers_gov":"hh_transfers"})

    df_income = df_person.merge(
        df_hh,
        on=["folioviv","foliohog"],
        how="left"
    )

    df_income["hh_transfers"] = df_income["hh_transfers"].fillna(0)

    # -----------------------------
    # SAVE
    # -----------------------------

    df_income.to_csv(
        PROCESSED_DIR / f"income_calculated_{year}.csv",
        index=False
    )


Processing income 2018

Processing income 2024


In [22]:
# =====================================================
# BLOQUE 2
# CALCULAR IVA
# =====================================================

import pandas as pd
import numpy as np

YEAR_CONFIG = {
    2018: {
        "iva_prefixes": ('C','D','F','G','H','I','J','K','L','M','N'),
        "informal_locs": [1,2,3,17]
    },
    2024: {
        "iva_prefixes": ('012','02','03','05','07','08','09','11','12'),
        "informal_locs": [1,2,3,17]
    }
}

def calculate_iva(year):

    config = YEAR_CONFIG[year]

    df_gastos = pd.read_csv(
        RAW_DIR / f"gastoshogar_{year}.csv",
        dtype={"folioviv":str,"foliohog":str,"clave":str},
        low_memory=False
    )

    df_gastos = df_gastos[df_gastos["tipo_gasto"] == "G1"].copy()

    df_gastos["gasto_tri"] = pd.to_numeric(
        df_gastos["gasto_tri"], errors="coerce"
    ).fillna(0)

    df_gastos["lleva_iva"] = df_gastos["clave"].str.startswith(
        config["iva_prefixes"]
    )

    df_gastos["es_formal"] = ~df_gastos["lugar_comp"].isin(
        config["informal_locs"]
    )

    factor_iva = 0.137931034

    df_gastos["iva_partida"] = np.where(
        df_gastos["lleva_iva"] & df_gastos["es_formal"],
        df_gastos["gasto_tri"] * factor_iva,
        0
    )

    iva_hh = df_gastos.groupby(
        ["folioviv","foliohog"]
    )["iva_partida"].sum().reset_index()

    iva_hh = iva_hh.rename(columns={"iva_partida":"IVA_HH"})

    return iva_hh

# Guardar resultados
iva_results = {}

for year in YEARS:
    iva_results[year] = calculate_iva(year)

In [23]:
# =====================================================
# BLOQUE 3
# MERGE FINAL + POST TAX
# =====================================================

import pandas as pd

all_years = []

for year in YEARS:

    print(f"\nBuilding dataset {year}")

    df_pob = pd.read_csv(
        RAW_DIR / f"poblacion_{year}.csv",
        dtype={"folioviv":"string","foliohog":"string","numren":"int64"}
    )[["folioviv","foliohog","numren","sexo","edad","parentesco"]]

    df_hog = pd.read_csv(
        RAW_DIR / f"concentradohogar_{year}.csv",
        dtype={"folioviv":"string","foliohog":"string"}
    )[["folioviv","foliohog","factor"]]

    df_income = pd.read_csv(
        PROCESSED_DIR / f"income_calculated_{year}.csv",
        dtype={"folioviv":"string","foliohog":"string","numren":"int64"}
    )

    df_iva = iva_results[year]

    # -----------------------------
    # MERGES
    # -----------------------------

    df = df_pob.merge(df_hog, on=["folioviv","foliohog"], how="left")

    df = df.merge(
        df_income,
        on=["folioviv","foliohog","numren"],
        how="left"
    )

    df = df.merge(
        df_iva,
        on=["folioviv","foliohog"],
        how="left"
    )

    df["IVA_HH"] = df["IVA_HH"].fillna(0)

    # -----------------------------
    # TAMAÑO HOGAR
    # -----------------------------

    df["hh_size"] = df.groupby(
        ["folioviv","foliohog"]
    )["numren"].transform("count")

    # -----------------------------
    # PER CÁPITA
    # -----------------------------

    df["pc_transfers"] = df["hh_transfers"] / df["hh_size"]
    df["pc_iva"] = df["IVA_HH"] / df["hh_size"]

    # -----------------------------
    # POST TAX INCOME
    # -----------------------------

    df["post_tax_income"] = (
        df["market_income"].fillna(0)
        + df["pc_transfers"]
        - df["pc_iva"]
    )

    df["year"] = year

    all_years.append(df)

# -----------------------------
# CONCAT
# -----------------------------

df_final = pd.concat(all_years, ignore_index=True)

df_final.to_csv(
    PROCESSED_DIR / "final_dataset_post_tax.csv",
    index=False
)

print("\nDataset final guardado")


Building dataset 2018


/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_41338/345444665.py:14: DtypeWarning: Columns (0: disc1, 1: segpop, 2: atemed, 3: peso) have mixed types. Specify dtype option on import or set low_memory=False.
  df_pob = pd.read_csv(



Building dataset 2024


/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_41338/345444665.py:14: DtypeWarning: Columns (0: conyuge_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df_pob = pd.read_csv(



Dataset final guardado
